In [1]:
# ============================================================
# COLAB CELL — Aggregate judge JSONL → per-model mean scores
# Works while the other notebook is still writing (reads whatever is available)
# ============================================================

import os, json
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

OUT_JSONL = "/content/drive/MyDrive/DL_Local/judge_eval_200__gemini3pro.jsonl"
assert os.path.exists(OUT_JSONL), f"❌ Not found: {OUT_JSONL}"

# ---- load JSONL (only status=ok) ----
rows = []
bad = 0
with open(OUT_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except Exception:
            bad += 1
            continue

        if obj.get("status") != "ok":
            continue

        sid = obj.get("id")
        scores = obj.get("scores", {})
        if not isinstance(scores, dict) or not scores:
            continue

        # scores: {model_name: {metric: int, ...}, ...}
        for model_name, msc in scores.items():
            if not isinstance(msc, dict):
                continue
            rec = {"sample_id": sid, "model": model_name}
            rec.update(msc)
            rows.append(rec)

print(f"✅ Loaded ok records: {len(set(r['sample_id'] for r in rows))} samples")
print(f"✅ Loaded model-score rows: {len(rows)} (should be samples*6)")
if bad:
    print(f"⚠️ Skipped unparsable lines (likely partial write while reading): {bad}")

if len(rows) == 0:
    raise RuntimeError("No ok records found yet. Wait for the other notebook to write some 'ok' lines.")

df = pd.DataFrame(rows)

# Ensure numeric
metric_cols = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
    "format_compliance",
]
for c in metric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# ---- aggregate means per model ----
agg = (
    df.groupby("model")[metric_cols]
      .mean()
      .sort_values(by=["decision_accuracy", "safety_score", "clinical_correctness", "completeness", "coherence"], ascending=False)
)

# Add counts
counts = df.groupby("model")["sample_id"].nunique().rename("n_samples")
agg = agg.join(counts)

# Optional: overall reasoning mean (3 subscores)
agg["reasoning_mean"] = agg[["clinical_correctness", "completeness", "coherence"]].mean(axis=1)

# Pretty rounding
agg_display = agg.copy()
for c in metric_cols + ["reasoning_mean"]:
    agg_display[c] = agg_display[c].round(3)

print("\n=== Per-model means (so far) ===")
display(agg_display)

# ---- extra: global progress summary ----
n_unique_samples = df["sample_id"].nunique()
print(f"\nProgress: judged {n_unique_samples}/200 samples ({n_unique_samples/200:.1%})")

# Optional: base vs instruct quick view
df["family"] = df["model"].str.replace(r"_(base|instruct)$", "", regex=True)
df["variant"] = df["model"].str.extract(r"_(base|instruct)$", expand=False)

pivot = (
    df.groupby(["family", "variant"])[metric_cols]
      .mean()
      .round(3)
)

print("\n=== Family × variant means (base vs instruct) ===")
display(pivot)


Mounted at /content/drive
✅ Loaded ok records: 200 samples
✅ Loaded model-score rows: 1200 (should be samples*6)

=== Per-model means (so far) ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,n_samples,reasoning_mean
model,,,,,,,,
qwen25_7b_instruct,4.375,3.855,3.305,3.760,4.725,1.260,200,3.930
qwen25_7b_base,4.300,3.800,3.105,3.295,4.690,4.085,200,3.697
qwen25_3b_base,3.825,3.255,2.795,3.060,4.385,3.945,200,3.413
qwen25_3b_instruct,3.375,3.100,2.015,3.170,4.205,1.195,200,3.130
qwen25_1p5b_instruct,3.050,2.255,1.160,2.555,3.795,0.945,200,2.503
qwen25_1p5b_base,2.550,2.090,1.335,2.390,3.660,2.245,200,2.462



Progress: judged 200/200 samples (100.0%)

=== Family × variant means (base vs instruct) ===


decision_accuracy  safety_score  clinical_correctness  \
family      variant                                                           
qwen25_1p5b base                  2.550         2.090                 1.335   
            instruct              3.050         2.255                 1.160   
qwen25_3b   base                  3.825         3.255                 2.795   
            instruct              3.375         3.100                 2.015   
qwen25_7b   base                  4.300         3.800                 3.105   
            instruct              4.375         3.855                 3.305   

                      completeness  coherence  format_compliance  
family      variant                                               
qwen25_1p5b base             2.390      3.660              2.245  
            instruct         2.555      3.795              0.945  
qwen25_3b   base             3.060      4.385              3.945  
            instruct         3.170      4.205              1.195  
qwen25_7b   base             3.295      4.690              4.085  
            instruct         3.760      4.725              1.260

In [2]:
# ============================================================
# COLAB CELL — Scan DL_Local for judge_eval__*__gemini.jsonl
# and render per-model mean tables (like your "upper table") for each file.
# ✅ Reads partial files safely (skips unparsable lines)
# ✅ Uses ONLY status=ok
# ✅ Prints how many samples were used per table
# ============================================================

import os, json, re, glob
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"

# Match your typical post-training naming:
# judge_eval__<experiment>__<N>__gemini.jsonl  (but we also accept any judge_eval*.jsonl)
patterns = [
    os.path.join(BASE_DIR, "judge_eval__*__gemini.jsonl"),
    os.path.join(BASE_DIR, "judge_eval__*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*.jsonl"),
]

files = []
seen = set()
for p in patterns:
    for fp in glob.glob(p):
        if fp not in seen and os.path.isfile(fp):
            seen.add(fp)
            files.append(fp)

files = sorted(files)

if not files:
    raise RuntimeError(f"No judge_eval*.jsonl files found under: {BASE_DIR}")

print("✅ Found judge files:", len(files))
for fp in files:
    print(" -", os.path.basename(fp))

metric_cols = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
    "format_compliance",
]

def load_ok_rows(jsonl_path: str):
    rows = []
    bad = 0
    ok_samples = set()

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue

            if obj.get("status") != "ok":
                continue

            sid = obj.get("id")
            scores = obj.get("scores", {})
            if not isinstance(scores, dict) or not scores:
                continue

            ok_samples.add(sid)

            # scores: {model_name: {metric: int, ...}, ...}
            for model_name, msc in scores.items():
                if not isinstance(msc, dict):
                    continue
                rec = {"sample_id": sid, "model": model_name}
                rec.update(msc)
                rows.append(rec)

    return rows, bad, len(ok_samples)

def per_model_means_table(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return None, None

    # Ensure numeric
    for c in metric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    agg = (
        df.groupby("model")[metric_cols]
          .mean()
          .sort_values(
              by=["decision_accuracy", "safety_score", "clinical_correctness", "completeness", "coherence"],
              ascending=False
          )
    )

    counts = df.groupby("model")["sample_id"].nunique().rename("n_samples")
    agg = agg.join(counts)

    agg["reasoning_mean"] = agg[["clinical_correctness", "completeness", "coherence"]].mean(axis=1)

    # Pretty rounding
    disp = agg.copy()
    for c in metric_cols + ["reasoning_mean"]:
        disp[c] = disp[c].round(3)

    n_unique = df["sample_id"].nunique()
    return disp, n_unique

def pretty_title_from_filename(fp: str) -> str:
    base = os.path.basename(fp)
    # Try to extract middle tag between judge_eval__ and __<N>__gemini
    m = re.match(r"judge_eval__([^_].*?)__\d+__gemini.*\.jsonl$", base)
    if m:
        return m.group(1)
    # fallback: filename without extension
    return os.path.splitext(base)[0]

# ---- Render one table per judge file ----
tables = []
for fp in files:
    rows, bad, n_ok_samples = load_ok_rows(fp)
    title = pretty_title_from_filename(fp)

    if len(rows) == 0:
        print("\n" + "="*90)
        print(f"📄 {title}")
        print("Path:", fp)
        print("✅ ok samples used:", n_ok_samples)
        if bad:
            print("⚠️ Skipped unparsable lines:", bad)
        print("⛔ No ok records found yet in this file.")
        continue

    table, n_unique = per_model_means_table(rows)

    print("\n" + "="*90)
    print(f"📄 {title}")
    print("Path:", fp)
    print(f"✅ ok samples used: {n_unique}")
    if bad:
        print(f"⚠️ Skipped unparsable lines (likely partial write while reading): {bad}")
    print("\n=== Per-model means (so far) ===")
    display(table)

    tables.append((title, fp, table, n_unique, bad))

print("\n✅ Done. Rendered tables:", len(tables))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found judge files: 10
 - judge_eval_200__gemini3pro.jsonl
 - judge_eval__e0_sft_adapted__100__gemini.jsonl
 - judge_eval__e1_wsft__100__gemini.jsonl
 - judge_eval__e2_cwsft_adapted__100__gemini.jsonl
 - judge_eval__e3_cwwsft__100__gemini.jsonl
 - judge_eval__e4_section_cw_wsft__100__gemini.jsonl
 - judge_eval__e5a_decision_entropy_sft__100__gemini.jsonl
 - judge_eval__e5b_explanation_entropy_sft__100__gemini.jsonl
 - judge_eval__e6_explanation_only_sft__100__gemini.jsonl
 - judge_eval__e7_decision_only_sft__100__gemini.jsonl

📄 judge_eval_200__gemini3pro
Path: /content/drive/MyDrive/DL_Local/judge_eval_200__gemini3pro.jsonl
✅ ok samples used: 200

=== Per-model means (so far) ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,n_samples,reasoning_mean
model,,,,,,,,
qwen25_7b_instruct,4.375,3.855,3.305,3.760,4.725,1.260,200,3.930
qwen25_7b_base,4.300,3.800,3.105,3.295,4.690,4.085,200,3.697
qwen25_3b_base,3.825,3.255,2.795,3.060,4.385,3.945,200,3.413
qwen25_3b_instruct,3.375,3.100,2.015,3.170,4.205,1.195,200,3.130
qwen25_1p5b_instruct,3.050,2.255,1.160,2.555,3.795,0.945,200,2.503
qwen25_1p5b_base,2.550,2.090,1.335,2.390,3.660,2.245,200,2.462



📄 e0_sft_adapted
Path: /content/drive/MyDrive/DL_Local/judge_eval__e0_sft_adapted__100__gemini.jsonl
✅ ok samples used: 100

=== Per-model means (so far) ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,n_samples,reasoning_mean
model,,,,,,,,
qwen25_7b_instruct,4.45,3.89,3.20,3.74,4.58,1.16,100,3.840
qwen25_3b_instruct,4.40,3.48,2.60,3.42,4.30,1.16,100,3.440
qwen25_1p5b_instruct,4.40,2.55,1.50,2.64,3.46,1.16,100,2.533
qwen25_7b_base,4.35,4.06,3.38,3.90,4.72,5.00,100,4.000
qwen25_3b_base,4.25,3.15,2.19,3.03,4.20,4.97,100,3.140
qwen25_1p5b_base,4.20,2.77,1.63,2.58,3.68,4.97,100,2.630



📄 e1_wsft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e1_wsft__100__gemini.jsonl
✅ ok samples used: 100

=== Per-model means (so far) ===


...


📄 e6_explanation_only_sft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e6_explanation_only_sft__100__gemini.jsonl
✅ ok samples used: 100

=== Per-model means (so far) ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,n_samples,reasoning_mean
model,,,,,,,,
qwen25_7b_instruct,4.55,4.24,3.64,4.08,4.65,1.26,100,4.123
qwen25_7b_base,4.50,4.20,3.48,4.10,4.85,4.99,100,4.143
qwen25_3b_base,4.40,3.72,3.03,3.68,4.68,4.99,100,3.797
qwen25_1p5b_base,4.15,3.12,2.50,2.72,4.14,0.32,100,3.120
qwen25_1p5b_instruct,3.50,2.37,1.36,2.35,3.29,0.07,100,2.333
qwen25_3b_instruct,3.45,3.08,2.26,3.27,4.16,1.01,100,3.230



📄 e7_decision_only_sft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e7_decision_only_sft__100__gemini.jsonl
✅ ok samples used: 99

=== Per-model means (so far) ===


,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance,n_samples,reasoning_mean
model,,,,,,,,
qwen25_3b_instruct,4.646,3.646,2.808,3.727,4.465,1.414,99,3.667
qwen25_1p5b_base,4.495,3.283,2.384,2.646,4.101,0.343,99,3.044
qwen25_7b_instruct,4.444,4.040,3.343,4.121,4.707,1.596,99,4.057
qwen25_7b_base,4.394,4.141,3.556,4.121,4.848,4.949,99,4.175
qwen25_3b_base,4.293,3.859,2.919,3.788,4.657,4.929,99,3.788
qwen25_1p5b_instruct,3.737,2.697,2.061,3.182,4.141,1.566,99,3.128



✅ Done. Rendered tables: 10


In [3]:
# ============================================================
# COLAB CELL — Scan DL_Local for judge_eval__*__gemini.jsonl
# and render DELTA-ONLY per-model mean tables vs a baseline file.
#
# ✅ Reads partial files safely (skips unparsable lines)
# ✅ Uses ONLY status=ok
# ✅ Prints how many samples were used (current + baseline)
# ✅ Delta per (model, metric): current_mean - baseline_mean
#
# EDIT:
#   BASELINE_JSONL = path to your "pre-training" judge jsonl (or any reference)
# ============================================================

import os, json, re, glob
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"

# 👇 Set your baseline (pre-training) evaluation file here
BASELINE_JSONL = "/content/drive/MyDrive/DL_Local/judge_eval_200__gemini3pro.jsonl"
assert os.path.exists(BASELINE_JSONL), f"❌ Baseline not found: {BASELINE_JSONL}"

patterns = [
    os.path.join(BASE_DIR, "judge_eval__*__gemini.jsonl"),
    os.path.join(BASE_DIR, "judge_eval__*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*.jsonl"),
]

files = []
seen = set()
for p in patterns:
    for fp in glob.glob(p):
        if fp not in seen and os.path.isfile(fp):
            seen.add(fp)
            files.append(fp)

files = sorted(files)
if not files:
    raise RuntimeError(f"No judge_eval*.jsonl files found under: {BASE_DIR}")

metric_cols = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
    "format_compliance",
]

def load_ok_rows(jsonl_path: str):
    rows = []
    bad = 0
    ok_samples = set()

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue

            if obj.get("status") != "ok":
                continue

            sid = obj.get("id")
            scores = obj.get("scores", {})
            if not isinstance(scores, dict) or not scores:
                continue

            ok_samples.add(sid)

            for model_name, msc in scores.items():
                if not isinstance(msc, dict):
                    continue
                rec = {"sample_id": sid, "model": model_name}
                rec.update(msc)
                rows.append(rec)

    return rows, bad, len(ok_samples)

def per_model_means(rows):
    df = pd.DataFrame(rows)
    if df.empty:
        return None, None

    for c in metric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    means = df.groupby("model")[metric_cols].mean()
    counts = df.groupby("model")["sample_id"].nunique().rename("n_samples")
    means = means.join(counts)

    means["reasoning_mean"] = means[["clinical_correctness", "completeness", "coherence"]].mean(axis=1)
    n_unique = df["sample_id"].nunique()
    return means, n_unique

def pretty_title_from_filename(fp: str) -> str:
    base = os.path.basename(fp)
    m = re.match(r"judge_eval__([^_].*?)__\d+__gemini.*\.jsonl$", base)
    if m:
        return m.group(1)
    return os.path.splitext(base)[0]

# ---- Baseline means ----
base_rows, base_bad, base_ok_samples = load_ok_rows(BASELINE_JSONL)
base_means, base_n_unique = per_model_means(base_rows)

if base_means is None or base_means.empty:
    raise RuntimeError("Baseline file has no ok records to compare against.")

# ---- Delta-only per file ----
print("✅ Found judge files:", len(files))
for fp in files:
    print(" -", os.path.basename(fp))

rendered = 0
for fp in files:
    if os.path.abspath(fp) == os.path.abspath(BASELINE_JSONL):
        continue

    rows, bad, n_ok_samples = load_ok_rows(fp)
    title = pretty_title_from_filename(fp)

    if len(rows) == 0:
        print("\n" + "="*90)
        print(f"📄 {title}")
        print("Path:", fp)
        print("⛔ No ok records found yet in this file.")
        if bad:
            print("⚠️ Skipped unparsable lines:", bad)
        continue

    cur_means, cur_n_unique = per_model_means(rows)
    if cur_means is None or cur_means.empty:
        continue

    # Align models
    common_models = sorted(set(cur_means.index).intersection(set(base_means.index)))
    if not common_models:
        print("\n" + "="*90)
        print(f"📄 {title}")
        print("Path:", fp)
        print("⛔ No overlapping model names with baseline (cannot compute deltas).")
        continue

    cur = cur_means.loc[common_models, metric_cols + ["reasoning_mean"]]
    base = base_means.loc[common_models, metric_cols + ["reasoning_mean"]]

    delta = (cur - base).round(3)
    delta = delta.rename(columns={c: f"Δ_{c}" for c in delta.columns})

    # Keep n_samples for context (NOT an "addition" metric; just sample count)
    delta = delta.join(cur_means.loc[common_models, ["n_samples"]])

    # Sort by delta decision_accuracy then delta safety etc (same spirit as original)
    delta = delta.sort_values(
        by=["Δ_decision_accuracy", "Δ_safety_score", "Δ_clinical_correctness", "Δ_completeness", "Δ_coherence"],
        ascending=False
    )

    print("\n" + "="*90)
    print(f"📄 {title}")
    print("Path:", fp)
    print(f"✅ ok samples used (current): {cur_n_unique} | (baseline): {base_n_unique}")
    if bad:
        print(f"⚠️ Skipped unparsable lines (likely partial write while reading): {bad}")
    if base_bad:
        print(f"⚠️ Baseline skipped unparsable lines: {base_bad}")

    print("\n=== Per-model DELTAS vs baseline (current - baseline) ===")
    display(delta)

    rendered += 1

print("\n✅ Done. Rendered delta tables:", rendered)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found judge files: 10
 - judge_eval_200__gemini3pro.jsonl
 - judge_eval__e0_sft_adapted__100__gemini.jsonl
 - judge_eval__e1_wsft__100__gemini.jsonl
 - judge_eval__e2_cwsft_adapted__100__gemini.jsonl
 - judge_eval__e3_cwwsft__100__gemini.jsonl
 - judge_eval__e4_section_cw_wsft__100__gemini.jsonl
 - judge_eval__e5a_decision_entropy_sft__100__gemini.jsonl
 - judge_eval__e5b_explanation_entropy_sft__100__gemini.jsonl
 - judge_eval__e6_explanation_only_sft__100__gemini.jsonl
 - judge_eval__e7_decision_only_sft__100__gemini.jsonl

📄 e0_sft_adapted
Path: /content/drive/MyDrive/DL_Local/judge_eval__e0_sft_adapted__100__gemini.jsonl
✅ ok samples used (current): 100 | (baseline): 200

=== Per-model DELTAS vs baseline (current - baseline) ===


,Δ_decision_accuracy,Δ_safety_score,Δ_clinical_correctness,Δ_completeness,Δ_coherence,Δ_format_compliance,Δ_reasoning_mean,n_samples
model,,,,,,,,
qwen25_1p5b_base,1.650,0.680,0.295,0.190,0.020,2.725,0.168,100
qwen25_1p5b_instruct,1.350,0.295,0.340,0.085,-0.335,0.215,0.030,100
qwen25_3b_instruct,1.025,0.380,0.585,0.250,0.095,-0.035,0.310,100
qwen25_3b_base,0.425,-0.105,-0.605,-0.030,-0.185,1.025,-0.273,100
qwen25_7b_instruct,0.075,0.035,-0.105,-0.020,-0.145,-0.100,-0.090,100
qwen25_7b_base,0.050,0.260,0.275,0.605,0.030,0.915,0.303,100



📄 e1_wsft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e1_wsft__100__gemini.jsonl
✅ ok samples used (current): 100 | (baseline): 200

=== Per-model DELTAS vs baseline (current - baseline) ===


,Δ_decision_accuracy,Δ_safety_score,Δ_clinical_correctness,Δ_completeness,Δ_coherence,Δ_format_compliance,Δ_reasoning_mean,n_samples
model,,,,,,,,
qwen25_1p5b_base,1.950,1.180,0.885,0.710,0.520,2.725,0.705,100
qwen25_1p5b_instruct,1.400,0.315,0.630,0.305,-0.035,0.025,0.300,100
qwen25_3b_instruct,1.175,0.430,0.605,0.430,0.235,-0.225,0.423,100
qwen25_3b_base,0.725,0.285,0.065,0.520,0.165,0.995,0.250,100
qwen25_7b_base,0.300,0.230,0.375,0.655,0.010,0.905,0.347,100
qwen25_7b_instruct,0.175,0.135,-0.085,0.170,-0.085,-0.290,0.000,100



📄 e2_cwsft_adapted
Path: /content/drive/MyDrive/DL_Local/judge_eval__e2_cwsft_adapted__100__gemini.jsonl
✅ ok samples used (current): 100 | (baseline): 200

=== Per-model DELTAS vs baseline (current - baseline) ===


...


📄 e6_explanation_only_sft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e6_explanation_only_sft__100__gemini.jsonl
✅ ok samples used (current): 100 | (baseline): 200

=== Per-model DELTAS vs baseline (current - baseline) ===


,Δ_decision_accuracy,Δ_safety_score,Δ_clinical_correctness,Δ_completeness,Δ_coherence,Δ_format_compliance,Δ_reasoning_mean,n_samples
model,,,,,,,,
qwen25_1p5b_base,1.600,1.030,1.165,0.330,0.480,-1.925,0.658,100
qwen25_3b_base,0.575,0.465,0.235,0.620,0.295,1.045,0.383,100
qwen25_1p5b_instruct,0.450,0.115,0.200,-0.205,-0.505,-0.875,-0.170,100
qwen25_7b_base,0.200,0.400,0.375,0.805,0.160,0.905,0.447,100
qwen25_7b_instruct,0.175,0.385,0.335,0.320,-0.075,0.000,0.193,100
qwen25_3b_instruct,0.075,-0.020,0.245,0.100,-0.045,-0.185,0.100,100



📄 e7_decision_only_sft
Path: /content/drive/MyDrive/DL_Local/judge_eval__e7_decision_only_sft__100__gemini.jsonl
✅ ok samples used (current): 99 | (baseline): 200

=== Per-model DELTAS vs baseline (current - baseline) ===


,Δ_decision_accuracy,Δ_safety_score,Δ_clinical_correctness,Δ_completeness,Δ_coherence,Δ_format_compliance,Δ_reasoning_mean,n_samples
model,,,,,,,,
qwen25_1p5b_base,1.945,1.193,1.049,0.256,0.441,-1.902,0.582,99
qwen25_3b_instruct,1.271,0.546,0.793,0.557,0.260,0.219,0.537,99
qwen25_1p5b_instruct,0.687,0.442,0.901,0.627,0.346,0.621,0.625,99
qwen25_3b_base,0.468,0.604,0.124,0.728,0.272,0.984,0.375,99
qwen25_7b_base,0.094,0.341,0.451,0.826,0.158,0.864,0.478,99
qwen25_7b_instruct,0.069,0.185,0.038,0.361,-0.018,0.336,0.127,99



✅ Done. Rendered delta tables: 9


In [4]:
# ============================================================
# COLAB CELL — Summary table: how many OK evaluations per judge file
# ============================================================

import os, json, glob
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"

patterns = [
    os.path.join(BASE_DIR, "judge_eval__*__gemini.jsonl"),
    os.path.join(BASE_DIR, "judge_eval__*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*.jsonl"),
]

files = []
seen = set()
for p in patterns:
    for fp in glob.glob(p):
        if fp not in seen and os.path.isfile(fp):
            seen.add(fp)
            files.append(fp)

files = sorted(files)
if not files:
    raise RuntimeError("No judge_eval*.jsonl files found.")

rows = []

for fp in files:
    ok_ids = set()
    total_ok_rows = 0
    bad = 0

    with open(fp, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue

            if obj.get("status") == "ok":
                total_ok_rows += 1
                sid = obj.get("id")
                if sid is not None:
                    ok_ids.add(sid)

    rows.append({
        "file": os.path.basename(fp),
        "ok_samples": len(ok_ids),      # unique sample IDs with status=ok
        "ok_rows": total_ok_rows,       # number of ok JSONL rows
        "bad_lines": bad
    })

df = pd.DataFrame(rows).sort_values("ok_samples", ascending=False)

print("\n=== OK evaluation count per judge file ===")
display(df.reset_index(drop=True))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== OK evaluation count per judge file ===


,file,ok_samples,ok_rows,bad_lines
0,judge_eval_200__gemini3pro.jsonl,200,200,0
1,judge_eval__e0_sft_adapted__100__gemini.jsonl,100,100,0
2,judge_eval__e1_wsft__100__gemini.jsonl,100,100,0
3,judge_eval__e2_cwsft_adapted__100__gemini.jsonl,100,100,0
4,judge_eval__e3_cwwsft__100__gemini.jsonl,100,100,0
5,judge_eval__e4_section_cw_wsft__100__gemini.jsonl,100,100,0
6,judge_eval__e5a_decision_entropy_sft__100__gem...,100,100,0
7,judge_eval__e5b_explanation_entropy_sft__100__...,100,100,0
8,judge_eval__e6_explanation_only_sft__100__gemi...,100,100,0
9,judge_eval__e7_decision_only_sft__100__gemini....,99,99,0


In [5]:
# ============================================================
# COLAB CELL — Method impact analysis (NO metric aggregation)
# Outputs ONLY:
#   Table 1) Overall: mean normalized gain across ALL models, per (method × metric)
#   Table 2) Base vs Instruct: normalized by BASE within each (method, metric)
#   Table 3) By size: THREE separate tables (1.5B / 3B / 7B), each is mean normalized gain across models of that size
#
# Notes:
# - No "OK evaluations per file" table
# - No Table 1b counts
# ============================================================

import os, json, re, glob
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"

# ---- EDIT: baseline pre-training file ----
BASELINE_JSONL = "/content/drive/MyDrive/DL_Local/judge_eval_200__gemini3pro.jsonl"
assert os.path.exists(BASELINE_JSONL), f"❌ Baseline not found: {BASELINE_JSONL}"

patterns = [
    os.path.join(BASE_DIR, "judge_eval__*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*.jsonl"),
]

metric_cols = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
    "format_compliance",
]

def pretty_method_name(fp: str) -> str:
    base = os.path.basename(fp)
    m = re.match(r"judge_eval__([^_].*?)__\d+__gemini.*\.jsonl$", base)
    if m:
        return m.group(1)
    return os.path.splitext(base)[0]

def parse_variant(model_name: str) -> str:
    m = re.search(r"_(base|instruct)$", model_name)
    return m.group(1) if m else "unknown"

def parse_size_bucket(model_name: str) -> str:
    m = re.search(r"_([0-9]+p?[0-9]*)b_", model_name)
    if not m:
        return "unknown"
    token = m.group(1).lower().replace("p", ".")
    try:
        val = float(token)
    except Exception:
        return "unknown"
    if abs(val - 1.5) < 1e-6:
        return "1.5B"
    if abs(val - 3.0) < 1e-6:
        return "3B"
    if abs(val - 7.0) < 1e-6:
        return "7B"
    return "unknown"

def load_ok_scores_long(jsonl_path: str, tag: str):
    rows = []
    bad = 0
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue
            if obj.get("status") != "ok":
                continue
            sid = obj.get("id")
            scores = obj.get("scores", {})
            if not isinstance(scores, dict) or not scores:
                continue
            for model_name, msc in scores.items():
                if not isinstance(msc, dict):
                    continue
                for metric in metric_cols:
                    if metric in msc:
                        rows.append({
                            "tag": tag,
                            "sample_id": sid,
                            "model": model_name,
                            "metric": metric,
                            "score": msc.get(metric, None),
                        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["score"] = pd.to_numeric(df["score"], errors="coerce")
    return df, bad

def mean_scores_by_model_metric(df_long: pd.DataFrame) -> pd.DataFrame:
    if df_long.empty:
        return pd.DataFrame(columns=["model", "metric", "mean_score"])
    means = (
        df_long.groupby(["model", "metric"])["score"]
               .mean()
               .rename("mean_score")
               .reset_index()
    )
    return means

def normalized_span_scores(delta_df: pd.DataFrame) -> pd.DataFrame:
    g = delta_df.groupby(["model", "metric"])["delta"]
    minv = g.transform("min")
    maxv = g.transform("max")
    span = (maxv - minv)
    eps = 1e-12
    out = delta_df.copy()
    out["norm_span"] = (out["delta"] - minv) / (span + eps)
    out.loc[span.abs() < 1e-12, "norm_span"] = float("nan")
    return out

# ---- Collect method files ----
files = []
seen = set()
for p in patterns:
    for fp in glob.glob(p):
        if fp not in seen and os.path.isfile(fp):
            seen.add(fp)
            files.append(fp)
files = sorted(files)

if not files:
    raise RuntimeError(f"No judge_eval*.jsonl files found under: {BASE_DIR}")

# ---- Load baseline ----
base_long, _ = load_ok_scores_long(BASELINE_JSONL, tag="baseline")
if base_long.empty:
    raise RuntimeError("Baseline has no status=ok records.")
base_mm = mean_scores_by_model_metric(base_long).rename(columns={"mean_score": "base_mean"})

# ---- Load all methods (post-training files) ----
method_dfs = []
for fp in files:
    if os.path.abspath(fp) == os.path.abspath(BASELINE_JSONL):
        continue
    method = pretty_method_name(fp)
    df_long, _ = load_ok_scores_long(fp, tag=method)
    if df_long.empty:
        continue
    mm = mean_scores_by_model_metric(df_long).rename(columns={"mean_score": "post_mean"})
    mm["method"] = method
    method_dfs.append(mm)

if not method_dfs:
    raise RuntimeError("No post-training judge files with status=ok records found yet.")

post_all = pd.concat(method_dfs, ignore_index=True)

# ---- Merge with baseline to compute deltas ----
merged = post_all.merge(base_mm[["model", "metric", "base_mean"]], on=["model", "metric"], how="inner")
merged["delta"] = merged["post_mean"] - merged["base_mean"]

# Add grouping metadata
merged["variant"] = merged["model"].apply(parse_variant)
merged["size"] = merged["model"].apply(parse_size_bucket)

# ---- Normalize within (model, metric) across methods ----
merged = normalized_span_scores(merged)

# ============================================================
# TABLE 1 — Overall mean normalized gain across ALL models
# ============================================================
t1 = (
    merged.groupby(["method", "metric"])["norm_span"]
          .mean()
          .unstack("metric")
          .reindex(columns=metric_cols)
          .round(3)
)

print("\n" + "="*90)
print("TABLE 1 — Overall mean normalized gain across ALL models (method × metric)")
display(t1)

# ============================================================
# TABLE 2 — Base vs Instruct, normalized by BASE (per method × metric)
# i.e., show (mean_instruct / mean_base) for each method/metric
# ============================================================
t2_means = (
    merged[merged["variant"].isin(["base", "instruct"])]
      .groupby(["method", "variant", "metric"])["norm_span"]
      .mean()
      .reset_index()
)

t2_pivot = t2_means.pivot_table(index=["method", "metric"], columns="variant", values="norm_span", aggfunc="mean")
t2_pivot["instruct_over_base"] = t2_pivot["instruct"] / (t2_pivot["base"] + 1e-12)

t2 = (
    t2_pivot["instruct_over_base"]
      .reset_index()
      .pivot(index="method", columns="metric", values="instruct_over_base")
      .reindex(columns=metric_cols)
      .round(3)
)

print("\n" + "="*90)
print("TABLE 2 — Instruct normalized by Base (instruct_mean / base_mean) per method × metric")
display(t2)

# ============================================================
# TABLE 3 — By size: THREE separate tables (1.5B / 3B / 7B)
# Each table: mean normalized gain across models of that size (method × metric)
# ============================================================
for sz in ["1.5B", "3B", "7B"]:
    sub = merged[merged["size"] == sz]
    if sub.empty:
        print("\n" + "="*90)
        print(f"TABLE 3 — Size {sz}: ⛔ no data (no overlapping models or no ok records)")
        continue

    t3 = (
        sub.groupby(["method", "metric"])["norm_span"]
           .mean()
           .unstack("metric")
           .reindex(columns=metric_cols)
           .round(3)
    )

    print("\n" + "="*90)
    print(f"TABLE 3 — Size {sz}: mean normalized gain across models (method × metric)")
    display(t3)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

TABLE 1 — Overall mean normalized gain across ALL models (method × metric)


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,0.323,0.193,0.164,0.138,0.216,0.750
e1_wsft,0.935,0.483,0.571,0.651,0.656,0.530
e2_cwsft_adapted,0.599,0.730,0.674,0.556,0.570,0.470
e3_cwwsft,0.987,0.740,0.480,0.426,0.622,0.366
e4_section_cw_wsft,0.585,0.459,0.549,0.506,0.598,0.752
e5a_decision_entropy_sft,0.816,0.879,0.487,0.659,0.295,0.506
e5b_explanation_entropy_sft,0.827,0.840,0.646,0.620,0.718,0.560
e6_explanation_only_sft,0.321,0.570,0.605,0.490,0.609,0.468
e7_decision_only_sft,0.417,0.739,0.827,0.855,0.956,0.501



TABLE 2 — Instruct normalized by Base (instruct_mean / base_mean) per method × metric


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,12.558,3.058,5.736,4.419,5.918,0.689
e1_wsft,1.085,0.773,0.626,0.885,1.221,0.609
e2_cwsft_adapted,0.985,0.909,0.925,1.321,1.254,0.241
e3_cwwsft,0.973,0.748,1.047,1.297,1.381,0.262
e4_section_cw_wsft,1.893,1.597,0.951,1.491,0.623,0.824
e5a_decision_entropy_sft,1.364,0.825,1.263,1.124,0.717,0.626
e5b_explanation_entropy_sft,1.368,0.974,1.251,0.971,0.987,0.361
e6_explanation_only_sft,1.077,0.414,0.380,0.436,0.258,0.557
e7_decision_only_sft,0.996,0.678,0.813,1.410,1.073,597.841



TABLE 3 — Size 1.5B: mean normalized gain across models (method × metric)


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,0.545,0.155,0.100,0.174,0.100,0.862
e1_wsft,1.000,0.644,0.646,0.807,0.766,0.799
e2_cwsft_adapted,0.805,0.714,0.606,0.701,0.633,0.733
e3_cwwsft,1.000,0.705,0.429,0.492,0.605,0.723
e4_section_cw_wsft,0.519,0.635,0.791,0.780,0.833,0.884
e5a_decision_entropy_sft,0.902,0.983,0.590,0.820,0.717,0.792
e5b_explanation_entropy_sft,0.759,0.972,0.653,0.778,0.811,0.768
e6_explanation_only_sft,0.000,0.330,0.500,0.135,0.451,0.000
e7_decision_only_sft,0.618,0.766,0.933,0.564,0.913,0.503



TABLE 3 — Size 3B: mean normalized gain across models (method × metric)


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,0.397,0.282,0.250,0.164,0.226,0.647
e1_wsft,0.888,0.592,0.664,0.724,0.816,0.259
e2_cwsft_adapted,0.632,0.775,0.705,0.565,0.660,0.358
e3_cwwsft,0.960,0.825,0.650,0.583,0.771,0.171
e4_section_cw_wsft,0.725,0.726,0.660,0.594,0.555,0.691
e5a_decision_entropy_sft,0.796,0.853,0.715,0.658,0.169,0.498
e5b_explanation_entropy_sft,0.888,0.923,0.881,0.715,0.623,0.439
e6_explanation_only_sft,0.214,0.402,0.500,0.429,0.500,0.700
e7_decision_only_sft,0.561,0.899,0.837,1.000,0.967,0.500



TABLE 3 — Size 7B: mean normalized gain across models (method × metric)


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,0.026,0.143,0.142,0.077,0.323,0.742
e1_wsft,0.917,0.214,0.404,0.421,0.385,0.531
e2_cwsft_adapted,0.360,0.700,0.712,0.402,0.417,0.320
e3_cwwsft,1.000,0.690,0.361,0.202,0.490,0.203
e4_section_cw_wsft,0.513,0.014,0.195,0.144,0.406,0.679
e5a_decision_entropy_sft,0.750,0.800,0.157,0.499,0.000,0.228
e5b_explanation_entropy_sft,0.833,0.624,0.406,0.368,0.719,0.472
e6_explanation_only_sft,0.750,0.976,0.816,0.905,0.875,0.702
e7_decision_only_sft,0.073,0.552,0.709,1.000,0.989,0.500


In [6]:
# ============================================================
# COLAB CELL — RAW DELTA table (post − baseline), aggregated across ALL models
# Outputs a table like Table 1: rows=method, cols=metrics
# (NOT normalized; positive = improves vs baseline, negative = harms vs baseline)
# ============================================================

import os, json, re, glob
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"

# ---- EDIT: baseline (pre-training) judge file ----
BASELINE_JSONL = "/content/drive/MyDrive/DL_Local/judge_eval_200__gemini3pro.jsonl"
assert os.path.exists(BASELINE_JSONL), f"❌ Baseline not found: {BASELINE_JSONL}"

# ---- Which post-training judge files to include as methods ----
patterns = [
    os.path.join(BASE_DIR, "judge_eval__*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*__gemini*.jsonl"),
    os.path.join(BASE_DIR, "judge_eval*.jsonl"),
]

metric_cols = [
    "decision_accuracy",
    "safety_score",
    "clinical_correctness",
    "completeness",
    "coherence",
    "format_compliance",
]

def pretty_method_name(fp: str) -> str:
    base = os.path.basename(fp)
    m = re.match(r"judge_eval__([^_].*?)__\d+__gemini.*\.jsonl$", base)
    if m:
        return m.group(1)
    return os.path.splitext(base)[0]

def load_ok_scores_long(jsonl_path: str, tag: str):
    rows = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue
            if obj.get("status") != "ok":
                continue
            sid = obj.get("id")
            scores = obj.get("scores", {})
            if not isinstance(scores, dict) or not scores:
                continue
            for model_name, msc in scores.items():
                if not isinstance(msc, dict):
                    continue
                for metric in metric_cols:
                    if metric in msc:
                        rows.append({
                            "tag": tag,
                            "sample_id": sid,
                            "model": model_name,
                            "metric": metric,
                            "score": msc.get(metric, None),
                        })
    df = pd.DataFrame(rows)
    if not df.empty:
        df["score"] = pd.to_numeric(df["score"], errors="coerce")
    return df

def mean_scores_by_model_metric(df_long: pd.DataFrame) -> pd.DataFrame:
    if df_long.empty:
        return pd.DataFrame(columns=["model", "metric", "mean_score"])
    return (
        df_long.groupby(["model", "metric"])["score"]
               .mean()
               .rename("mean_score")
               .reset_index()
    )

# ---- Collect method files ----
files = []
seen = set()
for p in patterns:
    for fp in glob.glob(p):
        if fp not in seen and os.path.isfile(fp):
            seen.add(fp)
            files.append(fp)
files = sorted(files)

if not files:
    raise RuntimeError(f"No judge_eval*.jsonl files found under: {BASE_DIR}")

# ---- Load baseline means ----
base_long = load_ok_scores_long(BASELINE_JSONL, tag="baseline")
if base_long.empty:
    raise RuntimeError("Baseline has no status=ok records.")
base_mm = mean_scores_by_model_metric(base_long).rename(columns={"mean_score": "base_mean"})

# ---- Load each method, compute model-level deltas, then average across models ----
all_rows = []

for fp in files:
    if os.path.abspath(fp) == os.path.abspath(BASELINE_JSONL):
        continue

    method = pretty_method_name(fp)
    post_long = load_ok_scores_long(fp, tag=method)
    if post_long.empty:
        continue

    post_mm = mean_scores_by_model_metric(post_long).rename(columns={"mean_score": "post_mean"})

    merged = post_mm.merge(base_mm[["model", "metric", "base_mean"]], on=["model", "metric"], how="inner")
    merged["delta"] = merged["post_mean"] - merged["base_mean"]
    merged["method"] = method

    # Average delta across models (per metric)
    agg = (
        merged.groupby(["method", "metric"])["delta"]
              .mean()
              .reset_index()
    )
    all_rows.append(agg)

if not all_rows:
    raise RuntimeError("No post-training judge files with status=ok records found yet.")

delta_all = pd.concat(all_rows, ignore_index=True)

# ---- Table like Table 1: method × metric ----
t_raw_delta = (
    delta_all.pivot(index="method", columns="metric", values="delta")
             .reindex(columns=metric_cols)
             .sort_index()
             .round(4)
)

print("\n" + "="*90)
print("RAW DELTA TABLE — mean(post − baseline) across ALL models (method × metric)")
print("Positive = improves vs baseline; Negative = harms vs baseline.")
display(t_raw_delta)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

RAW DELTA TABLE — mean(post − baseline) across ALL models (method × metric)
Positive = improves vs baseline; Negative = harms vs baseline.


metric,decision_accuracy,safety_score,clinical_correctness,completeness,coherence,format_compliance
method,,,,,,
e0_sft_adapted,0.7625,0.2575,0.1308,0.1800,-0.0867,0.7908
e1_wsft,0.9542,0.4292,0.4125,0.4650,0.1350,0.6892
e2_cwsft_adapted,0.8542,0.5275,0.4458,0.4017,0.0850,0.5958
e3_cwwsft,0.9708,0.5342,0.3358,0.3533,0.1050,0.5725
e4_section_cw_wsft,0.8375,0.4442,0.4175,0.4117,0.1167,0.8225
e5a_decision_entropy_sft,0.9042,0.6042,0.3958,0.4683,0.0183,0.6792
e5b_explanation_entropy_sft,0.9125,0.6042,0.4758,0.4500,0.1483,0.6458
e6_explanation_only_sft,0.5125,0.3958,0.4258,0.3283,0.0517,-0.1725
e7_decision_only_sft,0.7559,0.5519,0.5593,0.5593,0.2432,0.1872
